In [1]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Sweet Crumbs Orders")      # run once; next time use gc.open("Sweet Crumbs Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/12mJbdaLNu6P3rrmKmjWIj6z7cWbey58-aT3iooXsUqE


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")


In [3]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes will be Rs 1300. Let me know your name and pickup date to place the order.
All set, Ravi! Your 2 eggless chocolate cakes are booked for pickup on 24 Dec. Total: Rs 1300. We’ll see you then—happy baking!


In [5]:
for m in history:
    m.pretty_print()


================================ Human Message =================================

Are you open on Sunday? How much are 2 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_d3d6ddac-3091-4586-9f17-4146a219546b)
 Call ID: fc_d3d6ddac-3091-4586-9f17-4146a219546b
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_ca4d9c2a-b690-428c-886a-be56e88d256e)
 Call ID: fc_ca4d9c2a-b690-428c-886a-be56e88d256e
  Args:
    item: chocolate cake
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x chocolate cake = Rs 1300
================================== Ai Message ==================================

Yes! We’re open Sunday from 10 am to 6 pm. Two chocolate cakes will

In [6]:
print(chat("Do you have donuts?"))
print(chat("Book 2 plum cakes for Kiran."))
print(chat("Book 0 cupcakes for Kiran on 25 Dec."))

We don’t have donuts in our menu, but we do offer a variety of cakes, cupcakes, and pastries. Let me know if there’s something else you’d like, and I’ll be happy to help!
Sure thing, Kiran! When would you like to pick up the 2 plum cakes? (We need a pickup date—24 hours’ notice required.)
It looks like you’d like 0 cupcakes—does that mean you don’t want any, or was that a typo? Let me know the correct quantity and we’ll book it for Kiran on 25 Dec.


In [7]:
class Appointment(BaseModel):
    """A patient appointment booking for Surya Homoeo Clinic."""
    patient_name: str = Field(description="Patient's full or first name")
    branch: Literal["kakinada", "pithapuram"] = Field(description="Clinic branch city")
    day: str = Field(description="Preferred day or date of visit, e.g. 'Wednesday' or '28 Nov'")
    is_followup: bool = Field(description="True if this is a follow-up visit, False if new consultation")
    medicine_weeks: int = Field(description="Weeks of medicine requested", ge=0, default=1)

extractor = llm.with_structured_output(Appointment)

messy_messages = [
    "hi im suresh need appointment at kakinada branch on wednesday first time visit",
    "Anitha here, follow up visit at pithapuram this friday, need 2 weeks medicine",
    "book slot for ramesh in kakinada tomorrow new consultation 1 week meds",
    "hello doctor meena coming to pitapuram on saturday for followup no medicine needed",
    "can u book karthik at kakinada on monday 3 weeks medicine new patient"
]

for msg in messy_messages:
    print(extractor.invoke(msg))

patient_name='Suresh' branch='kakinada' day='Wednesday' is_followup=False medicine_weeks=1
patient_name='Anitha' branch='pithapuram' day='Friday' is_followup=True medicine_weeks=2
patient_name='ramesh' branch='kakinada' day='tomorrow' is_followup=False medicine_weeks=1
patient_name='Meena' branch='pithapuram' day='Saturday' is_followup=True medicine_weeks=0
patient_name='karthik' branch='kakinada' day='Monday' is_followup=False medicine_weeks=3


In [15]:
clinic_sh = gc.create("Surya Homoeo Appointments")   # run once; later use gc.open("Surya Homoeo Appointments")
clinic_ws = clinic_sh.sheet1
clinic_ws.clear()
clinic_ws.append_row([
    "patient_name",
    "branch",
    "day",
    "new_month",
    "medicines_for_month",
    "estimated_cost"
])
print("Updated sheet headers successfully!")

print("Clinic Sheet ready:", clinic_sh.url)

@tool
def log_appointment(patient_name: str, branch: str, day: str, is_followup: bool, medicine_weeks: int = 1) -> str:
    """Save a confirmed patient appointment to the Surya Homoeo Appointments sheet.
    Use only when patient_name, branch ('kakinada' or 'pithapuram'), day, and whether it is a follow-up are known."""
    try:
        appt = Appointment(
            patient_name=patient_name,
            branch=branch.lower().strip(),
            day=day,
            is_followup=is_followup,
            medicine_weeks=medicine_weeks
        )
    except Exception as e:
        return f"Could not save appointment: {e}. Ask the patient to clarify."

    consult_min, consult_max = (0, 0) if appt.is_followup else (200, 300)
    est_cost = f"Rs {consult_min + appt.medicine_weeks * 700}-{consult_max + appt.medicine_weeks * 800}"
    clinic_ws.append_row([
        appt.patient_name, appt.branch, appt.day,
        "yes" if appt.is_followup else "no", appt.medicine_weeks, est_cost
    ])
    return f"Booked: {appt.patient_name} at {appt.branch.title()} on {appt.day} ({est_cost})."

Updated sheet headers successfully!
Clinic Sheet ready: https://docs.google.com/spreadsheets/d/1PhVuVEHk-NXGlfcDiQnWSct1a0hTG5OPKMjJx1RLeSU


In [12]:
@tool
def estimate_clinic_cost(is_same_month_followup: bool, medicine_months: int = 1) -> str:
    """Estimate total cost in Rs for Surya Homoeo Clinic.
    Use when a patient asks about consultation fees, follow-up fees, or monthly medicine costs.

    Args:
        is_same_month_followup: True if the patient already visited within the current month (free consultation).
        medicine_months: Number of months of homeopathic medicine needed (default 1).
    """
    consult_min, consult_max = (0, 0) if is_same_month_followup else (200, 300)
    total_min = consult_min + medicine_months * 700
    total_max = consult_max + medicine_months * 800
    return (
        f"Estimated total: Rs {total_min} to Rs {total_max} "
        f"(monthly medicines usually cost Rs 700–800/month and depend on the patient's health problem)."
    )

@tool
def get_branch_timings(branch: str, day: str) -> str:
    """Check operating hours for Surya Homoeo Clinic at 'kakinada' or 'pithapuram' on a given day of the week."""
    b, d = branch.strip().lower(), day.strip().lower()
    if d == "sunday":
        return "Both Kakinada and Pithapuram branches are closed on Sundays."
    if b == "kakinada":
        return "Kakinada timings: 10:00 AM to 2:30 PM" if d == "wednesday" else "Kakinada timings: 4:00 PM to 9:30 PM"
    if b == "pithapuram":
        return "Pithapuram timings: 4:00 PM to 9:00 PM" if d == "wednesday" else "Pithapuram timings: 10:00 AM to 2:30 PM"
    return f"'{branch}' is not a valid branch. Available branches: Kakinada and Pithapuram."

In [16]:
CLINIC_SYSTEM = """ROLE:   You are the polite front-desk assistant for Surya Homoeo Clinic.
FACTS:  Branches in Kakinada and Pithapuram. Closed on Sundays. Same-month follow-ups are free.
RULES:  Always use get_branch_timings for hours and estimate_clinic_cost for fees; never guess.
        Never diagnose symptoms or suggest medicines; say "Consult doctor as soon as possible."
        Before log_appointment, collect patient_name, branch, day, and whether it is a follow-up.
FORMAT: Under 60 words, calm and polite, English or Telugu matching the patient."""

clinic_agent = create_agent(
    model=llm,
    tools=[get_branch_timings, estimate_clinic_cost, log_appointment],
    system_prompt=CLINIC_SYSTEM
)

res = clinic_agent.invoke({"messages": [{"role": "user", "content": "What are Kakinada timings on Wednesday, and book a new visit for Tharun on Wednesday with 1 week medicine."}]})

for m in res["messages"]:
    m.pretty_print()

================================ Human Message =================================

What are Kakinada timings on Wednesday, and book a new visit for Tharun on Wednesday with 1 week medicine.
================================== Ai Message ==================================
Tool Calls:
  get_branch_timings (fc_d4bf2196-0d53-4128-89ec-09428030b1f9)
 Call ID: fc_d4bf2196-0d53-4128-89ec-09428030b1f9
  Args:
    branch: kakinada
    day: Wednesday
================================= Tool Message =================================
Name: get_branch_timings

Kakinada timings: 10:00 AM to 2:30 PM
================================== Ai Message ==================================
Tool Calls:
  log_appointment (fc_121ab795-8e99-437a-987a-f405f48b8d47)
 Call ID: fc_121ab795-8e99-437a-987a-f405f48b8d47
  Args:
    branch: kakinada
    day: Wednesday
    is_followup: False
    medicine_weeks: 1
    patient_name: Tharun
================================= Tool Message =================================
Name: log_